# Web Scraping BBC — Dataset multi-classe (9 classi)

Articoli BBC suddivisi in **9 classi**:

| # | Label | Sezione BBC | Path URL |
|---|-------|-------------|----------|
| 1 | `football` | Football / Soccer | `/sport/football` |
| 2 | `cricket` | Cricket | `/sport/cricket` |
| 3 | `rugby` | Rugby Union | `/sport/rugby-union` |
| 4 | `formula1` | Formula 1 | `/sport/formula1` |
| 5 | `tennis` | Tennis | `/sport/tennis` |
| 6 | `golf` | Golf | `/sport/golf` |
| 7 | `american_football` | NFL / American Football | `/sport/american-football` |
| 8 | `other_sport` | Tutti gli altri sport | `/sport/*` residuale |
| 9 | `non_sport` | News, Culture, Business… | `/news`, `/culture`, … |

**Obiettivo**: provare a raggiungere 1 000 articoli per classe.

In [ ]:
import csv
import time
import random
import logging
from datetime import datetime
from urllib.parse import urljoin, urlparse
from collections import deque

import requests
from bs4 import BeautifulSoup

# Quanti articoli per classe vogliamo raccogliere
OBIETTIVO_PER_CLASSE = 1000

# File di output
OUTPUT_CSV = "bbc_dataset_9classi.csv"

# Pausa tra una richiesta e l'altra (in secondi)
PAUSA_MIN = 0.4
PAUSA_MAX = 0.9

# Header HTTP
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-GB,en;q=0.9",
}

TIMEOUT_RICHIESTA = 15

# Segmenti di URL da escludere (contenuti non testuali)
SEGMENTI_DA_ESCLUDERE = {
    "/video/", "/live/", "/programmes/", "/av/",
    "/sounds/", "/weather/", "/bitesize/", "/food/recipes/",
    "/mediaaction/", "/worklife/", "/reel/", "/iplayer/",
    "/archive/", "/cbbc/", "/cbeebies/", "/teach/",
    "/mundo/",
}

DOMINIO_CONSENTITO = "bbc.com"


# Definizione delle 9 classi
CLASSI = [
    {
        "label": "football",
        "seed_urls": [
            "https://www.bbc.com/sport/football",
            "https://www.bbc.com/sport/football/premier-league",
            "https://www.bbc.com/sport/football/champions-league",
            "https://www.bbc.com/sport/football/europa-league",
            "https://www.bbc.com/sport/football/english-football-league",
            "https://www.bbc.com/sport/football/scottish-football",
            "https://www.bbc.com/sport/football/womens-football",
            "https://www.bbc.com/sport/football/transfers",
        ],
        "path_prefixes": ["/sport/football"],
    },
    {
        "label": "cricket",
        "seed_urls": [
            "https://www.bbc.com/sport/cricket",
            "https://www.bbc.com/sport/cricket/english-cricket",
            "https://www.bbc.com/sport/cricket/international-cricket",
        ],
        "path_prefixes": ["/sport/cricket"],
    },
    {
        "label": "rugby",
        "seed_urls": [
            "https://www.bbc.com/sport/rugby-union",
            "https://www.bbc.com/sport/rugby-union/six-nations",
            "https://www.bbc.com/sport/rugby-union/english-premiership",
        ],
        "path_prefixes": ["/sport/rugby-union", "/sport/rugby-league"],
    },
    {
        "label": "formula1",
        "seed_urls": [
            "https://www.bbc.com/sport/formula1",
        ],
        "path_prefixes": ["/sport/formula1"],
    },
    {
        "label": "tennis",
        "seed_urls": [
            "https://www.bbc.com/sport/tennis",
        ],
        "path_prefixes": ["/sport/tennis"],
    },
    {
        "label": "golf",
        "seed_urls": [
            "https://www.bbc.com/sport/golf",
        ],
        "path_prefixes": ["/sport/golf"],
    },
    {
        "label": "american_football",
        "seed_urls": [
            "https://www.bbc.com/sport/american-football",
        ],
        "path_prefixes": ["/sport/american-football"],
    },
    {
        "label": "other_sport",
        "seed_urls": [
            "https://www.bbc.com/sport",
            "https://www.bbc.com/sport/athletics",
            "https://www.bbc.com/sport/cycling",
            "https://www.bbc.com/sport/swimming",
            "https://www.bbc.com/sport/boxing",
            "https://www.bbc.com/sport/motorsport",
            "https://www.bbc.com/sport/horse-racing",
            "https://www.bbc.com/sport/snooker",
            "https://www.bbc.com/sport/darts",
            "https://www.bbc.com/sport/disability-sport",
        ],
        # other_sport è il catch-all: qualsiasi /sport/* non coperto sopra
        "path_prefixes": ["/sport"],
    },
    {
        "label": "non_sport",
        "seed_urls": [
            "https://www.bbc.com/news",
            "https://www.bbc.com/news/world",
            "https://www.bbc.com/news/business",
            "https://www.bbc.com/news/technology",
            "https://www.bbc.com/news/science_and_environment",
            "https://www.bbc.com/news/entertainment_and_arts",
            "https://www.bbc.com/news/health",
            "https://www.bbc.com/news/education",
            "https://www.bbc.com/culture",
            "https://www.bbc.com/travel",
            "https://www.bbc.com/future",
        ],
        "path_prefixes": [
            "/news", "/culture", "/travel", "/future",
            "/business", "/innovation",
            "/technology", "/science", "/entertainment",
            "/health", "/education", "/world",
        ],
    },
]

# Prefissi "specifici" delle classi sport nominate. Servono per distinguere
# uno sport specifico da other_sport nel classificatore URL
SPORT_SPECIFICI_PREFISSI = set()
for classe in CLASSI:
    if classe["label"] not in ("other_sport", "non_sport"):
        for p in classe["path_prefixes"]:
            if p.startswith("/sport"):
                SPORT_SPECIFICI_PREFISSI.add(p)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
def pausa():
    """Attende un tempo casuale tra PAUSA_MIN e PAUSA_MAX secondi."""
    time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))


def scarica_pagina(url: str) -> BeautifulSoup | None:
    """
    Scarica una pagina web e restituisce un oggetto BeautifulSoup.
    Restituisce None se la richiesta fallisce o il content-type non è HTML.
    """
    try:
        risposta = requests.get(url, headers=HEADERS, timeout=TIMEOUT_RICHIESTA)
        risposta.raise_for_status()
        content_type = risposta.headers.get("Content-Type", "")
        if "text/html" not in content_type:
            return None
        return BeautifulSoup(risposta.text, "html.parser")
    except requests.RequestException as errore:
        log.debug("Errore nel download di %s: %s", url, errore)
        return None

In [ ]:

def e_url_valido(url: str) -> bool:
    """
    Verifica che un URL sia interno a bbc.com, non contenga segmenti
    esclusi e abbia un path sufficientemente lungo.
    """
    parsed = urlparse(url)
    if parsed.scheme not in ("http", "https"):
        return False
    if not parsed.hostname or DOMINIO_CONSENTITO not in parsed.hostname:
        return False

    percorso = parsed.path.lower()
    for segmento in SEGMENTI_DA_ESCLUDERE:
        if segmento in percorso:
            return False
    if len(percorso.strip("/")) < 5:
        return False
    if parsed.fragment:
        return False
    return True


def e_url_articolo(url: str) -> bool:
    """
    Un articolo BBC contiene 'articles'/'article' nel path,
    oppure l'ultimo segmento è alfanumerico lungo (formato vecchio).
    """
    percorso = urlparse(url).path.lower().rstrip("/")
    segmenti = percorso.split("/")

    if "articles" in segmenti or "article" in segmenti:
        return True

    if segmenti:
        ultimo = segmenti[-1]
        if any(c.isdigit() for c in ultimo) and len(ultimo) > 8:
            return True
    return False


def classifica_url(url: str) -> str | None:
    """
    Classifica un URL in una delle 9 classi in base al suo path.
    """
    percorso = urlparse(url).path.lower()

    # 1. Sport specifici (ordine: dal più specifico al meno specifico)
    for classe in CLASSI:
        if classe["label"] in ("other_sport", "non_sport"):
            continue
        for prefix in classe["path_prefixes"]:
            if percorso.startswith(prefix):
                return classe["label"]

    # 2. Sport generico → other_sport
    if percorso.startswith("/sport"):
        return "other_sport"

    # 3. Non-sport
    classe_non_sport = next(c for c in CLASSI if c["label"] == "non_sport")
    for prefix in classe_non_sport["path_prefixes"]:
        if percorso.startswith(prefix):
            return "non_sport"

    return None

In [ ]:
def estrai_articolo(soup: BeautifulSoup) -> tuple[str, str] | None:
    """
    Estrae di titolo (H1) e di corpo del testo pulito.
    Restituisce (titolo, testo) oppure None.
    """
    # ── TITOLO ──
    tag_h1 = soup.find("h1")
    if not tag_h1:
        return None
    titolo = tag_h1.get_text(strip=True)
    if len(titolo) < 10:
        return None

    # ── CORPO ──
    articolo = soup.find("article")
    if not articolo:
        return None

    # Rimozione elementi non testuali
    tag_da_rimuovere = [
        "nav", "footer", "aside", "figure", "figcaption",
        "form", "button", "script", "style", "svg", "iframe",
        "video", "audio",
    ]
    for tag_name in tag_da_rimuovere:
        for elemento in articolo.find_all(tag_name):
            elemento.decompose()

    # Rimozione div promozionali / navigazione
    for div in articolo.find_all("div"):
        if div.attrs is None:
            continue
        ruolo = div.get("role", "").lower()
        classi = " ".join(div.get("class", [])).lower()
        data_comp = div.get("data-component", "").lower()
        if any(kw in classi for kw in ("related", "promo", "billboard", "ad-slot",
                                        "share", "social", "newsletter", "banner",
                                        "cookie", "consent", "topic-list")):
            div.decompose()
        elif ruolo in ("navigation", "banner", "complementary"):
            div.decompose()
        elif data_comp in ("related-stories", "tag-list", "links-block"):
            div.decompose()

    # Raccolta di paragrafi
    paragrafi = articolo.find_all("p")
    frasi = [p.get_text(strip=True) for p in paragrafi if len(p.get_text(strip=True)) > 40]
    testo_completo = "\n".join(frasi)

    if len(testo_completo) < 200:
        return None

    return titolo, testo_completo

In [ ]:
def estrai_link(soup: BeautifulSoup, url_base: str) -> list[str]:
    """
    Estrae tutti i link <a href> dalla pagina, li rende assoluti
    e restituisce solo quelli validi.
    """
    link_trovati = []
    for tag_a in soup.find_all("a", href=True):
        href = tag_a["href"]
        url_assoluto = urljoin(url_base, href)
        parsed = urlparse(url_assoluto)
        url_pulito = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
        if e_url_valido(url_pulito):
            link_trovati.append(url_pulito)
    return link_trovati

In [ ]:
def raccogli_articoli(
    seed_urls: list[str],
    etichetta_target: str,
    obiettivo: int,
    url_globali_visitati: set,
) -> list[dict]:
    """
    Crawling BFS a partire dai seed URL.
    Raccoglie fino a `obiettivo` articoli classificati come `etichetta_target`.
    """
    coda = deque(seed_urls)
    articoli_raccolti = []
    pagine_analizzate = 0
    inizio = time.time()

    log.info("▶ Inizio crawling per classe '%s' (obiettivo: %d)", etichetta_target, obiettivo)

    while coda and len(articoli_raccolti) < obiettivo:
        url_corrente = coda.popleft()

        if url_corrente in url_globali_visitati:
            continue
        url_globali_visitati.add(url_corrente)

        pausa()

        soup = scarica_pagina(url_corrente)
        if soup is None:
            continue

        pagine_analizzate += 1

        if e_url_articolo(url_corrente):
            classe = classifica_url(url_corrente)
            if classe == etichetta_target:
                risultato = estrai_articolo(soup)
                if risultato:
                    titolo, testo = risultato
                    articoli_raccolti.append({
                        "title": titolo,
                        "text": testo,
                        "label": etichetta_target,
                        "url": url_corrente,
                    })

                    n = len(articoli_raccolti)
                    if n % 100 == 0:
                        trascorso = time.time() - inizio
                        velocita = n / trascorso * 60  
                        eta_min = (obiettivo - n) / velocita if velocita > 0 else 0
                        log.info(
                            "   [%s] %d/%d | %.0f art/min | ETA ~%.0f min | coda: %d",
                            etichetta_target, n, obiettivo,
                            velocita, eta_min, len(coda),
                        )

        nuovi_link = estrai_link(soup, url_corrente)
        for link in nuovi_link:
            if link not in url_globali_visitati:
                classe_link = classifica_url(link)
                if classe_link == etichetta_target:
                    coda.append(link)

        if pagine_analizzate % 200 == 0:
            log.info(
                "   [%s] Pagine: %d | Coda: %d | Articoli: %d/%d",
                etichetta_target, pagine_analizzate, len(coda),
                len(articoli_raccolti), obiettivo,
            )

    trascorso = time.time() - inizio
    log.info(
        "✔ '%s' completato: %d articoli in %d pagine (%.1f min)",
        etichetta_target, len(articoli_raccolti), pagine_analizzate, trascorso / 60,
    )
    return articoli_raccolti

In [ ]:
def salva_csv(articoli: list[dict], percorso_file: str):
    """Salva la lista di articoli in CSV """
    with open(percorso_file, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "text", "label", "url"])
        writer.writeheader()
        writer.writerows(articoli)
    log.info("💾 Dataset salvato in '%s' (%d righe).", percorso_file, len(articoli))

In [ ]:
log.info("=" * 65)
log.info("  BBC Web Scraper — Dataset 9 classi")
log.info("  Classi: %s", ", ".join(c['label'] for c in CLASSI))
log.info("  Obiettivo: %d articoli per classe", OBIETTIVO_PER_CLASSE)
log.info("=" * 65)

url_visitati = set()

tutti_articoli = []

for classe_config in CLASSI:
    articoli = raccogli_articoli(
        seed_urls=classe_config["seed_urls"],
        etichetta_target=classe_config["label"],
        obiettivo=OBIETTIVO_PER_CLASSE,
        url_globali_visitati=url_visitati,
    )
    tutti_articoli.extend(articoli)

random.shuffle(tutti_articoli)

salva_csv(tutti_articoli, OUTPUT_CSV)

10:30:05  [INFO]  =================================================================
10:30:05  [INFO]    BBC Web Scraper — Dataset 9 classi
10:30:05  [INFO]    Classi: football, cricket, rugby, formula1, tennis, golf, american_football, other_sport, non_sport
10:30:05  [INFO]    Obiettivo: 1000 articoli per classe
10:30:05  [INFO]  =================================================================
10:30:05  [INFO]  ▶ Inizio crawling per classe 'football' (obiettivo: 1000)
10:33:31  [INFO]     [football] 100/1000 | 29 art/min | ETA ~31 min | coda: 2526
10:34:08  [INFO]     [football] Pagine: 200 | Coda: 2944 | Articoli: 122/1000
10:38:19  [INFO]     [football] Pagine: 400 | Coda: 7609 | Articoli: 140/1000
10:39:46  [INFO]     [football] 200/1000 | 21 art/min | ETA ~39 min | coda: 7833
10:42:13  [INFO]     [football] Pagine: 600 | Coda: 8037 | Articoli: 297/1000
10:42:20  [INFO]     [football] 300/1000 | 25 art/min | ETA ~29 min | coda: 8034
10:45:16  [INFO]     [football] 400/1000 | 26 ar

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
print("\n" + "=" * 50)
print("  RIEPILOGO DATASET")
print("=" * 50)
print(f"\nTotale articoli: {len(df)}")
print(f"\nDistribuzione per classe:")
print(df["label"].value_counts().to_string())
print(f"\nLunghezza media testo per classe (caratteri):")
print(df.groupby("label")["text"].apply(lambda x: x.str.len().mean()).round(0).to_string())
print("\n" + "=" * 50)


  RIEPILOGO DATASET

Totale articoli: 7049

Distribuzione per classe:
label
tennis               1000
other_sport          1000
rugby                1000
cricket              1000
non_sport            1000
football             1000
formula1              462
american_football     333
golf                  254

Lunghezza media testo per classe (caratteri):
label
american_football    3669.0
cricket              2289.0
football             2553.0
formula1             4502.0
golf                 3600.0
non_sport            5854.0
other_sport          2969.0
rugby                3089.0
tennis               3428.0

